# Models exploration

In [112]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression

from birdclef_2026_ml.notebooks.notebook_utils import init_notebook
from birdclef_2026_ml.paths import PATHS

from birdclef_2026_ml.processing.data_split import split_audio_train_val
from birdclef_2026_ml.models.one_vs_rest_models import *
from birdclef_2026_ml.models.eval import (
    evaluate_multiclass,
    plot_confusion_matrix,
    evaluate_soundscapes_multilabel
)

from birdclef_2026_ml.feature_engineering.configs import FeatureConfig, PoolingConfig, ChunkConfig, MILConfig
from birdclef_2026_ml.models import (
    build_mil_bags_from_df,
    train_one_vs_rest_model,
    predict_proba_one_vs_rest,
)
from birdclef_2026_ml.models.mil_learning import build_mil_bags_from_df
from birdclef_2026_ml.feature_engineering import build_feature_matrix_and_labels_from_df
from birdclef_2026_ml.models.hierarchical_models import (
    SoftCombinationArtifacts,
    predict_proba_soft_combination
)

init_notebook()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [85]:
train = pd.read_parquet(PATHS["proc_train"])
soundscapes = pd.read_parquet(PATHS["proc_soundscapes"])
taxonomy = pd.read_csv(PATHS["taxonomy"])
soundscapes.head(5)

,filename,start,end,primary_label,primary_label_list,datetime,start_sec,end_sec
0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380,"[22961, 23158, 24321, 517063, 65380]",2021-12-31 20:15:00,0.0,5.0
1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380,"[22961, 23158, 24321, 517063, 65380]",2021-12-31 20:15:00,5.0,10.0
2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380,"[22961, 23158, 24321, 517063, 65380]",2021-12-31 20:15:00,10.0,15.0
3,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:15,00:00:20,22961;23158;24321;517063;65380,"[22961, 23158, 24321, 517063, 65380]",2021-12-31 20:15:00,15.0,20.0
4,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:20,00:00:25,22961;23158;24321;517063;65380,"[22961, 23158, 24321, 517063, 65380]",2021-12-31 20:15:00,20.0,25.0


### Train/test split + features extraction

In [72]:
df_train, df_val = split_audio_train_val(train)

In [75]:
# Define features for models
feat_cfg = FeatureConfig(include_waveform_stats=False,
                         include_mfcc=True,
                         include_delta=False, 
                         include_delta2=False,
                         include_energy=False,
                         include_spectral=False)
pool_cfg = PoolingConfig()
chunk_cfg = ChunkConfig()

#### Usage without MIL

In [78]:
# Build it
X_train, y_train, features = build_feature_matrix_and_labels_from_df(
    df=df_train.head(10).reset_index(drop=True),
    label_cols=["class_name", "primary_label"],
    feature_cfg=feat_cfg,
    pooling_mode="chunk",
    pooling_cfg=pool_cfg,
    chunk_cfg=chunk_cfg,
)

X_val, y_val, _ = build_feature_matrix_and_labels_from_df(
    df=df_val.head(10).reset_index(drop=True),
    label_cols=["class_name", "primary_label"],
    feature_cfg=feat_cfg,
    pooling_mode="chunk",
    pooling_cfg=pool_cfg,
    chunk_cfg=chunk_cfg,
)

X_train.shape, X_val.shape

((43, 140), (72, 140))

In [ ]:
# Save it
# matrix_dir = PATHS["proc_train_matrix"]

# # Save X_train, y_train, features
# joblib.dump({
#     "X_train": X_train,
#     "y_train": y_train,
#     "features": features,
# }, matrix_dir / "train_mfcc.joblib")

# # Save X_val, y_val
# joblib.dump({
#     "X_val": X_val,
#     "y_val": y_val,
# }, matrix_dir / "val_mfcc.joblib")

# print(f"Train/val data and features saved to: {matrix_dir}")

['/Users/bsh2022/Study/master_ue/ml/birdclef_2026_ML/data/processed/matrices/train_mfcc.joblib']

In [ ]:
# Or import it
# data_train = joblib.load(matrix_dir / "train_mfcc.joblib")
# data_val = joblib.load(matrix_dir / "val_mfcc.joblib")

# X_train, y_train, features = data_train["X_train"], data_train["y_train"], data_train["features"]
# X_val, y_val = data_val["X_val"], data_val["y_val"]

#### Usage with MIL

In [ ]:
chunk_cfg = ChunkConfig(chunk_size_s=5.0, step_size_s=5.0)
mil_cfg = MILConfig(window_size_s=1.0, step_size_s=0.5, pooling="max")
bags_train, y_train, feature_names = build_mil_bags_from_df(
    df=df_train.head(25),
    label_cols="class_name",
    feature_cfg=feat_cfg,
    pooling_cfg=pool_cfg,
    chunk_cfg=chunk_cfg,
    mil_cfg=mil_cfg,
)

base_estimator = LogisticRegression(
    solver="liblinear",
    max_iter=1000,
    class_weight="balanced",
)

# 1) Train one-vs-rest models
class_name_ovr_artifacts = train_one_vs_rest_model(
    x=bags_train,
    y=y_train,
    estimator=base_estimator,
    n_jobs=-1,
    verbose=0,
    mil_mode=True,
    mil_config=mil_cfg,
)

/Users/bsh2022/Study/master_ue/ml/birdclef_2026_ML/.venv/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 0 is present in all training examples.
  warnings.warn(


In [111]:
# Explode, merge, then collapse class_name lists per soundscape
ss = soundscapes.reset_index().explode(["primary_label_list"])
ss = ss.merge(taxonomy, left_on="primary_label_list", right_on="primary_label")
ss = ss.groupby("index").agg({
    **{col: "first" for col in ss.columns if col not in ["class_name", "primary_label_list", "index"]},
    "primary_label_list": lambda x: list(x),
    "class_name": lambda x: list(np.unique(x))
}).reset_index()
ss.head()

,index,filename,start,end,primary_label_x,datetime,start_sec,end_sec,primary_label_y,inat_taxon_id,scientific_name,common_name,primary_label_list,class_name
0,0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380,2021-12-31 20:15:00,0.0,5.0,22961,22961,Leptodactylus podicipinus,Pointedbelly Frog,"[22961, 23158, 24321, 517063, 65380]",[Amphibia]
1,1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380,2021-12-31 20:15:00,5.0,10.0,22961,22961,Leptodactylus podicipinus,Pointedbelly Frog,"[22961, 23158, 24321, 517063, 65380]",[Amphibia]
2,2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380,2021-12-31 20:15:00,10.0,15.0,22961,22961,Leptodactylus podicipinus,Pointedbelly Frog,"[22961, 23158, 24321, 517063, 65380]",[Amphibia]
3,3,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:15,00:00:20,22961;23158;24321;517063;65380,2021-12-31 20:15:00,15.0,20.0,22961,22961,Leptodactylus podicipinus,Pointedbelly Frog,"[22961, 23158, 24321, 517063, 65380]",[Amphibia]
4,4,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:20,00:00:25,22961;23158;24321;517063;65380,2021-12-31 20:15:00,20.0,25.0,22961,22961,Leptodactylus podicipinus,Pointedbelly Frog,"[22961, 23158, 24321, 517063, 65380]",[Amphibia]


In [97]:
taxonomy[taxonomy["primary_label"] == "517063"]

,primary_label,inat_taxon_id,scientific_name,common_name,class_name
57,517063,517063,Pithecopus azureus,Southern Orange-legged Leaf Frog,Amphibia


In [115]:
evaluate_soundscapes_multilabel(ss.head(10),
                                class_name_ovr_artifacts,
                                feature_cfg=feat_cfg,
                                pooling_cfg=pool_cfg,
                                target_name="class_name",
                                label_col="class_name", 
                                mil_cfg=mil_cfg)

segment_proba [[1.]]
segment_proba [[1.]]
segment_proba [[1.]]
segment_proba [[1.]]
segment_proba [[1.]]
segment_proba [[1.]]
segment_proba [[1.]]
segment_proba [[1.]]
segment_proba [[1.]]
segment_proba [[1.]]


ValueError: Samplewise metrics are not available outside of multilabel classification.